In [1]:
"""
Gmail Email Categorizer
========================
Reads emails from Gmail for the past month, categorizes them by subject keywords,
assigns priority levels, and outputs a formatted DataFrame (and optional Excel file).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ONE-TIME SETUP (do this before running the script)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

STEP 1 — Install required libraries:
    pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client pandas openpyxl

STEP 2 — Create a Google Cloud Project & enable Gmail API:
    1. Go to https://console.cloud.google.com/
    2. Click "Select a project" → "New Project" → name it anything (e.g. "Email Categorizer") → Create
    3. In the search bar, search for "Gmail API" → click it → click "Enable"
    4. In the left menu go to: APIs & Services → OAuth consent screen
        - Choose "External" → Create
        - Fill in App name (anything), your Gmail as support email, and at the bottom as developer email
        - Click Save and Continue through the rest (no need to add scopes manually here)
    5. In the left menu go to: APIs & Services → Credentials
        - Click "+ Create Credentials" → "OAuth client ID"
        - Application type: "Desktop app" → Name it anything → Create
        - Click "Download JSON" on the popup — save this file as credentials.json
        - Put credentials.json in the SAME FOLDER as this script

STEP 3 — Run this script:
    The first time you run it, a browser window will open asking you to log in to Google
    and grant permission. After that, a token.json file is saved so you won't need to
    log in again.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

import os
import re
import base64
import pandas as pd
from datetime import datetime, timedelta, timezone

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build


# ──────────────────────────────────────────────
# CONFIGURATION — Edit these rules as needed
# ──────────────────────────────────────────────

CATEGORY_RULES = [
    {
        "category": "Cancelled",
        "priority": 1,
        # Any of these patterns (case-insensitive) will match
        "patterns": [
            r"cancel",      # catches: cancelled, cancellation, canceling, etc.
        ],
    },
    {
        "category": "Comm Delay",
        "priority": 2,
        "patterns": [
            r"comm(unication)?\s*(delay|delays|delayed)",
            r"communication\s*delay",
        ],
    },
    # ── Add more categories below as needed ──
    # {
    #     "category": "Outage",
    #     "priority": 3,
    #     "patterns": [r"outage", r"power\s*out"],
    # },
]

# How many days back to search
DAYS_BACK = 30

# Set to True to also save the result as an Excel file
SAVE_TO_EXCEL = True
EXCEL_OUTPUT_PATH = "email_categories.xlsx"

# OAuth scopes — read-only is all we need
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

# File paths (keep credentials.json in the same folder as this script)
CREDENTIALS_FILE = "credentials.json"
TOKEN_FILE = "token.json"




In [3]:
# ──────────────────────────────────────────────
# HELPER FUNCTIONS
# ──────────────────────────────────────────────

def get_gmail_service():
    """
    Handles OAuth authentication and returns an authorized Gmail API service object.
    On first run, opens a browser for login. After that, uses the saved token.json.
    """
    creds = None

    # Load saved token if it exists
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)

    # If no valid credentials, prompt login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not os.path.exists(CREDENTIALS_FILE):
                raise FileNotFoundError(
                    f"\n\nERROR: '{CREDENTIALS_FILE}' not found!\n"
                    "Please follow the setup steps at the top of this script to download\n"
                    "your credentials.json from Google Cloud Console.\n"
                )
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)

        # Save the token for next time
        with open(TOKEN_FILE, "w") as token:
            token.write(creds.to_json())

    return build("gmail", "v1", credentials=creds)


def categorize_subject(subject: str):
    """
    Returns (priority, category) if the subject matches a rule, else (None, None).
    Rules are checked in order — first match wins.
    """
    for rule in CATEGORY_RULES:
        for pattern in rule["patterns"]:
            if re.search(pattern, subject, re.IGNORECASE):
                return rule["priority"], rule["category"]
    return None, None


def parse_date(date_str: str):
    """
    Parses Gmail's date string (e.g. 'Thu, 12 Feb 2026 10:30:00 -0600')
    into a Python datetime object.
    """
    # Strip timezone name in parentheses if present, e.g. "(CST)"
    date_str = re.sub(r"\s*\(.*\)\s*$", "", date_str.strip())
    formats = [
        "%a, %d %b %Y %H:%M:%S %z",
        "%d %b %Y %H:%M:%S %z",
        "%a, %d %b %Y %H:%M:%S",
        "%d %b %Y %H:%M:%S",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None


def format_dates(dt: datetime):
    """Returns (date_sent, month_sent) formatted strings."""
    if dt is None:
        return "Unknown", "Unknown"
    date_sent = f"{dt.month}/{dt.day}/{str(dt.year)[2:]}"   # e.g. 2/12/26
    month_sent = dt.strftime("%b-%y")                        # e.g. Feb-26
    return date_sent, month_sent


def get_header(headers: list, name: str) -> str:
    """Pulls a specific header value (like 'Subject' or 'Date') from the headers list."""
    for h in headers:
        if h["name"].lower() == name.lower():
            return h["value"]
    return ""


# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────

def main():
    # Connect to Gmail
    print("Connecting to Gmail...")
    service = get_gmail_service()
    print("Connected!\n")

    # Build date filter for Gmail query (Gmail uses yyyy/mm/dd format)
    cutoff_date = (datetime.now() - timedelta(days=DAYS_BACK)).strftime("%Y/%m/%d")
    query = f"after:{cutoff_date}"

    print(f"Fetching emails from the past {DAYS_BACK} days...")

    # Get list of matching message IDs
    results = service.users().messages().list(
        userId="me",
        q=query,
        maxResults=500       # Increase if you have a very high email volume
    ).execute()

    messages = results.get("messages", [])

    if not messages:
        print("No emails found in that date range.")
        return

    print(f"Found {len(messages)} emails. Categorizing...\n")

    rows = []
    for msg_ref in messages:
        try:
            # Fetch just the metadata (subject + date) — much faster than full message
            msg = service.users().messages().get(
                userId="me",
                id=msg_ref["id"],
                format="metadata",
                metadataHeaders=["Subject", "Date"]
            ).execute()

            headers = msg.get("payload", {}).get("headers", [])
            subject = get_header(headers, "Subject") or "(No Subject)"
            date_str = get_header(headers, "Date")

            priority, category = categorize_subject(subject)

            # Skip emails that don't match any category
            if priority is None:
                continue

            dt = parse_date(date_str)
            date_sent, month_sent = format_dates(dt)

            rows.append({
                "Priority":            priority,
                "Category":            category,
                "Header Name":         subject,
                "Date Sent mm-dd-yy":  date_sent,
                "Month Sent mm-yy":    month_sent,
            })

        except Exception as e:
            print(f"  Skipped a message due to error: {e}")
            continue

    if not rows:
        print("No emails matched any of the configured categories.")
        return

    # Build DataFrame and sort by Priority then Date
    df = pd.DataFrame(rows, columns=[
        "Priority", "Category", "Header Name",
        "Date Sent mm-dd-yy", "Month Sent mm-yy"
    ])
    df.sort_values(by=["Priority", "Date Sent mm-dd-yy"], ascending=[True, False], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Print to console
    print(f"\n{'='*80}")
    print(f"  EMAIL CATEGORY REPORT  |  Past {DAYS_BACK} days  |  {len(df)} matches")
    print(f"{'='*80}")
    print(df.to_string(index=False))
    print(f"{'='*80}\n")

    # Optionally save to Excel
    if SAVE_TO_EXCEL:
        with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name="Email Categories")

            # Auto-fit column widths
            ws = writer.sheets["Email Categories"]
            for col in ws.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)

        print(f"Results saved to: {EXCEL_OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Connecting to Gmail...
Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=997264419918-9vi0c975qlbfcl98kqef7cn190tkcerg.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A50717%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=vhjfXta29hWjiKW81wElSHGGZhQMZ0&access_type=offline
Connected!

Fetching emails from the past 30 days...
Found 500 emails. Categorizing...


  EMAIL CATEGORY REPORT  |  Past 30 days  |  2 matches
 Priority  Category                                Header Name Date Sent mm-dd-yy Month Sent mm-yy
        1 Cancelled Cancelled Disconnect FA due to a Reconnect            2/24/26           Feb-26
        1 Cancelled       RPA-811 Cancelled DFNP Summary | PRD            2/24/26           Feb-26

Results saved to: email_categories.xlsx
